# Hierarchical Teams: Delegated Ownership with Bounded Depth

| Field | Value |
|---|---|
| Stage | Multi-agent RAG |
| Difficulty | Advanced |
| Status | Complete |
| Requires network/API | No |
| Last reviewed | 2026-09-25 |

Callout - Key idea:
A hierarchy should mirror real team boundaries and cap delegation depth; extra layers are not free reasoning.

## 30-Second Summary

A root supervisor delegates a policy question to a policy-team lead, which invokes retrieval and verification specialists before returning a typed team result.

## Why This Matters

Hierarchies isolate tools and ownership across teams, but excessive layers amplify latency, context loss, and unclear failure responsibility.

## Scope

| Covers | Does not cover |
|---|---|
| Root/team roles, depth budget, retrieval-verification split, result contract | Dynamic organization design, distributed queues, live LLM teams |


## Mental Model

```text
root supervisor -> policy lead -> retriever -> verifier -> policy lead -> root
```


In [1]:
DOCUMENT = {"source": "security-policy", "text": "Production access requires manager approval."}
request = {"intent": "policy", "question": "Who approves production access?", "depth": 0, "max_depth": 2}


## How It Works

The root chooses a team, the team lead owns specialist sequencing, and the root accepts only a verified team result. Depth counts delegation layers, not internal function calls.


## Baseline

A flat root-to-retriever path finds text but has no independent verification status or team ownership.


In [2]:
baseline = {"evidence": DOCUMENT, "verified": False, "team": None}
baseline


{'evidence': {'source': 'security-policy',
  'text': 'Production access requires manager approval.'},
 'verified': False,
 'team': None}

## Technique Implementation

The root delegates one level; the team lead coordinates retrieval and verification within the final allowed depth.


In [3]:
def policy_team(state: dict) -> dict:
    assert state["depth"] <= state["max_depth"]
    evidence = DOCUMENT
    verified = "manager approval" in evidence["text"].lower()
    return {"team": "policy", "claim": "A manager approves production access.", "source": evidence["source"], "verified": verified, "depth": state["depth"]}

def root_supervisor(state: dict) -> dict:
    assert state["depth"] < state["max_depth"]
    delegated = {**state, "depth": state["depth"] + 1}
    return policy_team(delegated)

team_result = root_supervisor(request)
team_result


{'team': 'policy',
 'claim': 'A manager approves production access.',
 'source': 'security-policy',
 'verified': True,
 'depth': 1}

## Controlled Experiment

We compare verification and ownership while asserting the delegation depth stays within the declared limit.


In [4]:
answer = f"{team_result['claim']} [{team_result['source']}]" if team_result["verified"] else "Insufficient verified evidence."
experiment = {"baseline_verified": baseline["verified"], "hierarchy_verified": team_result["verified"], "within_depth": team_result["depth"] <= request["max_depth"], "answer": answer}
experiment


{'baseline_verified': False,
 'hierarchy_verified': True,
 'within_depth': True,
 'answer': 'A manager approves production access. [security-policy]'}

## Evaluation

The hierarchy adds explicit policy-team ownership and verification within **depth 1 of 2**. It is warranted only if that boundary reflects real governance.


In [5]:
assert not experiment["baseline_verified"] and experiment["hierarchy_verified"]
assert experiment["within_depth"] and team_result["team"] == "policy"
assert "[security-policy]" in answer
print("Hierarchical-team checks passed.")


Hierarchical-team checks passed.


## Decision Guide

| Need | Pattern |
|---|---|
| Separate teams/tool domains | Hierarchy |
| One stable dispatcher | Supervisor |
| Peer handoffs | Network |
| Single team and simple task | Avoid hierarchy |


## Failure Modes and Debugging

| Symptom | Cause | Fix |
|---|---|---|
| Context loss | Too many summaries | Typed result schema/provenance |
| Deep recursion | No depth budget | Hard max depth |
| Slow answer | Serial layers | Remove unjustified layer |
| Unclear failure owner | Delegation without status | Team-level terminal reason |


## Production Notes

### Observability
Trace root/team task IDs, depth, specialist status, evidence IDs, verification, and failure owner.

### Safety and Guardrails
Tool and tenant permissions remain scoped to each team.

### Latency and Cost
Charge a layer budget and execute independent specialists concurrently.


## Practice

Add a compliance team and make the root reject cross-team results without both team signatures.

## Recall

Toggle - Recall: When is a hierarchy useful?
When real teams or permission domains need delegated ownership.

Toggle - Recall: What bounds it?
A delegation-depth budget and typed team results.

## Sources

- [LangGraph hierarchical agent teams](https://langchain-ai.github.io/langgraph/tutorials/multi_agent/hierarchical_agent_teams/)
- Repository-owned synthetic policy fixture

## Review Log

| Date | Status | Confidence | Next review focus |
|---|---|---|---|
| 2026-09-25 | Complete; executed and visually reviewed | High for the bounded delegation example | Add multi-team parallel aggregation |
